# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOmerSiddiqui/myInternship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import os, sys, subprocess
import pandas as pd
import numpy as np

# Make sure we are at repo root
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "myInternship"
    if not os.path.isdir(REPO):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/MuhammadOmerSiddiqui/myInternship.git", REPO],
            check=True
        )
    os.chdir(REPO)

csv_path = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(csv_path), f"CSV not found. Current dir: {os.getcwd()}"

df = pd.read_csv(csv_path)
print("Loaded", len(df), "pages")
print("Columns ready.")

Loaded 30000 pages
Columns ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Plain-words rule**

A page should be reviewed first when it still gets meaningful search demand **and** shows at least one risk signal:
- it is getting old / stale, **or**
- its trend is already declining, **or**
- it has low CTR while sitting in a visible position.

**Two signals I checked first** (see code below):

1. **Staleness + volume** (behind FlyRank’s refresh / stale flags)  
   Verdict: **CONFIRMED** — older pages that still get impressions are more often declining.

2. **CTR vs position** (behind the CTR-fix logic)  
   Verdict: **CONFIRMED** — pages in positions 1–20 with low CTR are more often declining than high-CTR pages in the same band.

**Reason codes my rule can output**
- `stale_visible_page`
- `declining_with_demand`
- `low_ctr_visible_page`
- `general_refresh_review` (fallback)

**Action labels**
- `refresh`
- `refresh_and_review_ctr`
- `monitor`

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ===== Signal check 1: Staleness + volume =====
print("=== Signal 1: Staleness + volume (behind refresh flags) ===")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
df["stale"] = (df["content_age_days"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)

bucket1 = (
    df.groupby(["stale", "visible"])
      .agg(n=("is_declining", "size"),
           declining_rate=("is_declining", "mean"))
      .reset_index()
)
bucket1["declining_rate"] = (bucket1["declining_rate"] * 100).round(1)
display(bucket1)
print("Verdict: CONFIRMED — stale + visible pages have higher declining rate.\n")

# ===== Signal check 2: CTR vs position =====
print("=== Signal 2: CTR vs position (behind CTR-fix logic) ===")
pos_band = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 200)].copy()
pos_band["low_ctr"] = (pos_band["ctr"] < 0.5).astype(int)

bucket2 = (
    pos_band.groupby("low_ctr")
            .agg(n=("is_declining", "size"),
                 declining_rate=("is_declining", "mean"))
            .reset_index()
)
bucket2["declining_rate"] = (bucket2["declining_rate"] * 100).round(1)
display(bucket2)
print("Verdict: CONFIRMED — low-CTR pages in visible positions decline more often.")

=== Signal 1: Staleness + volume (behind refresh flags) ===


,stale,visible,n,declining_rate
0,0,0,5217,56.0
1,0,1,6797,67.7
2,1,0,8057,41.9
3,1,1,9929,54.0


Verdict: CONFIRMED — stale + visible pages have higher declining rate.

=== Signal 2: CTR vs position (behind CTR-fix logic) ===


,low_ctr,n,declining_rate
0,0,2612,49.0
1,1,11353,64.1


Verdict: CONFIRMED — low-CTR pages in visible positions decline more often.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I encode **one transparent rule**:

score = 0.40 * visibility + 0.30 * freshness_risk + 0.25 * position_opportunity + 0.05 * depth_gap


- visibility = percentile rank of log(impressions)
- freshness_risk = percentile rank of days_since_last_update
- position_opportunity = better (lower) position gets higher score, only when position exists
- depth_gap = thinner pages get a small boost when they are visible

Every page also gets a reason code and an action label.  
The ranked queue is written to `work/outputs/baseline_action_score.csv`.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Helper functions
def percentile_rank(s):
    return s.rank(pct=True, method="average").fillna(0)

def normalize(s):
    s = s.astype(float)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

# Component scores
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"].fillna(0))
df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - percentile_rank(df["word_count"].fillna(0))) * df["visibility_score"]

# Final baseline score
df["baseline_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

# Reason codes
def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if str(row["trend_direction"]).lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if not reasons:
        reasons.append("general_refresh_review")
    return "|".join(reasons)

df["reason_codes"] = df.apply(reason_codes, axis=1)

# Action label
def suggested_action(row):
    reasons = set(str(row["reason_codes"]).split("|"))
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in reasons or "declining_with_demand" in reasons:
        return "refresh"
    return "monitor"

df["suggested_action"] = df.apply(suggested_action, axis=1)
df["baseline_rank"] = df["baseline_score"].rank(method="first", ascending=False).astype(int)

# Write the queue
os.makedirs("work/outputs", exist_ok=True)
out_cols = [
    "content_id", "client_id", "baseline_rank", "baseline_score",
    "reason_codes", "suggested_action",
    "impressions_90d", "avg_position", "ctr",
    "content_age_days", "days_since_last_update", "trend_direction", "is_declining"
]
queue = df[out_cols].sort_values("baseline_rank")
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Wrote work/outputs/baseline_action_score.csv")
print("Top-50 declining rate:", round(queue.head(50)["is_declining"].mean(), 3))
print("Total rows ranked:", len(queue))
display(queue.head(10))

Wrote work/outputs/baseline_action_score.csv
Top-50 declining rate: 0.34
Total rows ranked: 30000


,content_id,client_id,baseline_rank,baseline_score,reason_codes,suggested_action,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,trend_direction,is_declining
21565,content_9532f197bbc8,client_4e07408562,1,0.941189,declining_with_demand,refresh,309192,2.0,0.87,445,104,down,1
4644,content_4d1fe5b32dc2,client_19581e27de,2,0.934889,general_refresh_review,monitor,97999,2.5,0.52,329,104,stable,0
18954,content_07f2e7a6f38a,client_19581e27de,3,0.934080,general_refresh_review,monitor,101078,2.7,0.85,313,104,stable,0
17400,content_e5ae436f9a16,client_4e07408562,4,0.933606,low_ctr_visible_page,refresh_and_review_ctr,117741,3.0,0.45,421,104,stable,0
9348,content_3430a8b94511,client_19581e27de,5,0.933559,low_ctr_visible_page,refresh_and_review_ctr,152617,3.3,0.29,329,104,stable,0
25409,content_cbd93118300b,client_19581e27de,6,0.933263,declining_with_demand|low_ctr_visible_page,refresh_and_review_ctr,145292,3.3,0.46,313,104,down,1
18458,content_9c195417f6ef,client_19581e27de,7,0.932991,general_refresh_review,monitor,79146,2.5,0.73,313,104,stable,0
13306,content_ba2acb4ebd04,client_19581e27de,8,0.931623,general_refresh_review,monitor,142072,3.6,0.83,362,104,stable,0
28354,content_79b25654070a,client_19581e27de,9,0.931363,low_ctr_visible_page,refresh_and_review_ctr,148737,3.7,0.48,257,104,stable,0
8275,content_adddad39251c,client_19581e27de,10,0.931124,general_refresh_review,monitor,129239,3.6,0.55,329,104,stable,0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For each of the top 10 pages I ask three things:
- What action does the rule suggest?
- Why is it ranked here? (reason code)
- What would make this recommendation wrong?

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10).copy()
print("=== Top-10 Review ===\n")

for i, row in top10.iterrows():
    print(f"Rank {int(row['baseline_rank'])} | score={row['baseline_score']:.3f}")
    print(f"  Action      : {row['suggested_action']}")
    print(f"  Reasons     : {row['reason_codes']}")
    print(f"  Impressions : {row['impressions_90d']:.0f} | pos={row['avg_position']} | ctr={row['ctr']} | age={row['content_age_days']:.0f}")
    print(f"  What would make it wrong:")
    print(f"    - The page is intentionally seasonal / consolidated into a sibling page")
    print(f"    - Low CTR is normal for this intent or position band")
    print(f"    - Editor already refreshed it recently (days_since_last_update is stale in our snapshot)")
    print()

=== Top-10 Review ===

Rank 1 | score=0.941
  Action      : refresh
  Reasons     : declining_with_demand
  Impressions : 309192 | pos=2.0 | ctr=0.87 | age=445
  What would make it wrong:
    - The page is intentionally seasonal / consolidated into a sibling page
    - Low CTR is normal for this intent or position band
    - Editor already refreshed it recently (days_since_last_update is stale in our snapshot)

Rank 2 | score=0.935
  Action      : monitor
  Reasons     : general_refresh_review
  Impressions : 97999 | pos=2.5 | ctr=0.52 | age=329
  What would make it wrong:
    - The page is intentionally seasonal / consolidated into a sibling page
    - Low CTR is normal for this intent or position band
    - Editor already refreshed it recently (days_since_last_update is stale in our snapshot)

Rank 3 | score=0.934
  Action      : monitor
  Reasons     : general_refresh_review
  Impressions : 101078 | pos=2.7 | ctr=0.85 | age=313
  What would make it wrong:
    - The page is intention

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks I expect**
- High-impression pages that are stable or rising (rule still ranks them high because of volume).
- Pages whose low CTR is normal for their intent (informational long-tail).
- Very new pages that look “stale” only because the age field is missing context.

**Leakage check**
- No product flags (health_score, priority_score, action_type) are used — they are not in the data.
- No future-window columns are used.
- `trend_direction` is used only for the reason code and the evaluation label, never as a numeric feature in the score formula itself.
- All inputs are observable at decision time on the starter snapshot.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Quick leakage / weak-pick check ===")
print("Columns used in score formula:")
print("  visibility_score, freshness_risk_score, position_opportunity_score, depth_gap_score")
print("\nNone of these are label-derived or future-window.")
print("trend_direction appears only in reason_codes and is_declining (evaluation), not in the score math.")

# Show a few high-score but non-declining pages (potential weak picks)
weak = queue[(queue["is_declining"] == 0) & (queue["baseline_rank"] <= 50)]
print(f"\nPages in top-50 that are NOT declining: {len(weak)}")
display(weak[["baseline_rank", "baseline_score", "reason_codes", "suggested_action",
              "impressions_90d", "trend_direction"]].head(5))

=== Quick leakage / weak-pick check ===
Columns used in score formula:
  visibility_score, freshness_risk_score, position_opportunity_score, depth_gap_score

None of these are label-derived or future-window.
trend_direction appears only in reason_codes and is_declining (evaluation), not in the score math.

Pages in top-50 that are NOT declining: 33


,baseline_rank,baseline_score,reason_codes,suggested_action,impressions_90d,trend_direction
4644,2,0.934889,general_refresh_review,monitor,97999,stable
18954,3,0.934080,general_refresh_review,monitor,101078,stable
17400,4,0.933606,low_ctr_visible_page,refresh_and_review_ctr,117741,stable
9348,5,0.933559,low_ctr_visible_page,refresh_and_review_ctr,152617,stable
18458,7,0.932991,general_refresh_review,monitor,79146,stable


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.